In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from datasets import load_dataset, DatasetDict
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("mteb/imdb")

# ds = DatasetDict({
#     "train": ds["train"].shuffle(seed=42).select(range(500)),
#     "test": ds["test"].shuffle(seed=42).select(range(200)),
# })

In [3]:
from transformers import TrainingArguments, Trainer
import time
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import os
from transformers import BitsAndBytesConfig
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_target_modules(model_id):
    model_id = model_id.lower()
    if "roberta" in model_id:
        return ["query", "value"]
    elif "deberta" in model_id:
        return ["query_proj", "value_proj"]
    elif "distilbert" in model_id:
        return ["q_lin", "v_lin"]
    else:
        # Default fallback for most BERT-like models
        return ["query", "value"]


def prepare_datasets(model_name, ds):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    
    tokenized_dataset = ds.map(tokenize_function, batched=True)
    
    cols_to_remove = [col for col in tokenized_dataset["train"].column_names if col not in ["input_ids", "attention_mask", "label"]]
    tokenized_dataset = tokenized_dataset.remove_columns(cols_to_remove)
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")
    
    train_valid = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)
    
    train_ds = train_valid["train"]
    val_ds = train_valid["test"]
    test_ds = tokenized_dataset["test"] 

    return train_ds, val_ds, test_ds



In [4]:
logs = []
def training_pipeline_fft(model_id, run_name,seed, train_ds, val_ds, test_ds):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        use_safetensors=True
    )
    set_seed(seed)

    print(f"--- Training Run: {run_name} ---")
    for param in model.parameters():
        param.requires_grad = True
    
    batch_size = 16
    learning_rate = 2e-5

    safe_model_name = model_id.split("/")[-1]
    out_dir = f"./results/{safe_model_name}_{run_name}_seed{seed}"
    args = TrainingArguments(
        output_dir=out_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=batch_size,
        bf16=True,
        num_train_epochs=2,
        logging_steps=10,
        load_best_model_at_end=True,
        label_names=["labels"],
    )

    trainer = Trainer(
        model,
        args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )
    
    os.system("nvidia-smi")
    start = time.time()
    trainer.train()
    os.system("nvidia-smi")
    end = time.time()
    train_time = end - start
    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_vram = 0
    predictions = trainer.predict(test_ds)

    preds = np.argmax(predictions.predictions, axis=1)
    
    labels = predictions.label_ids
    
    acc = accuracy_score(labels, preds)
        
    f1 = f1_score(labels, preds, average="macro")

    print(f"[{run_name}] Accuracy: {acc:.4f}")
    print(f"[{run_name}] F1 Score: {f1:.4f}")
    print(f"[{run_name}] Training Time: {train_time:.2f} seconds\n")

    logs.append({
        "seed": seed,
        "lr": learning_rate,
        "run_name": run_name,
        "accuracy": acc,
        "f1": f1,
        "train_time": train_time,
        "vram": peak_vram
    })


    del model, trainer
    torch.cuda.empty_cache()


In [5]:

model_ids = ["FacebookAI/roberta-base", "microsoft/deberta-v3-base", "distilbert/distilbert-base-uncased"]


for model_id in model_ids:
    train_ds, val_ds, test_ds = prepare_datasets(model_id, ds)
    for seed in [42, 43]:
        run_name = "full_fine_tuning"
        training_pipeline_fft(model_id, run_name, seed, train_ds, val_ds, test_ds)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: full_fine_tuning ---


Epoch,Training Loss,Validation Loss
1,0.684554,0.182203
2,0.572033,0.172484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,1.024693,0.242350
2,0.755999,0.246894


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


[full_fine_tuning] Accuracy: 0.9072
[full_fine_tuning] F1 Score: 0.9072
[full_fine_tuning] Training Time: 237.01 seconds



In [6]:
df = pd.DataFrame(logs)
df.to_csv("full_ft_experiment_results.csv", index=False)